In [1]:
# Import instaloader package
import instaloader

# creating an Instaloader() object
ig=instaloader.Instaloader()

# Taking the instagram username as input from user
usrname="fitwtanishq"

#Fetching the details of provided useraname using instaloder object
profile=instaloader.Profile.from_username(ig.context, usrname)

# Printing the fetched details and storing the profile pic of that account
print("Username: ", profile.username)
print("Number of Posts Uploaded: ", profile.mediacount)
print(profile.username+" is having " + str(profile.followers)+' followers.')
print(profile.username+" is following " + str(profile.followees)+' people')
print("Bio: ", profile.biography)
instaloader.Instaloader().download_profile(usrname,profile_pic_only=True)

JSON Query to api/v1/users/web_profile_info/: 429 Too Many Requests when accessing https://www.instagram.com/api/v1/users/web_profile_info/?username=fitwtanishq [retrying; skip with ^C]
Number of requests within last 10/11/20/22/30/60 minutes grouped by type:
 *                            other:    1    1    1    1    1    1
Instagram responded with HTTP error "429 - Too Many Requests". Please
do not run multiple instances of Instaloader in parallel or within
short sequence. Also, do not use any Instagram App while Instaloader
is running.
The request will be retried in 666 seconds, at 19:55.
[skipped by user]


ConnectionException: JSON Query to api/v1/users/web_profile_info/: 429 Too Many Requests when accessing https://www.instagram.com/api/v1/users/web_profile_info/?username=fitwtanishq

### contact extraction

In [1]:
import requests
import json

# Username of the profile to fetch
usrname = "fitwtanishq"
url = f"https://www.instagram.com/{usrname}/?__a=1&__d=dis"

# Send a GET request to fetch profile JSON
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    try:
        # Extracting contact information if available
        business_contact = data['graphql']['user'].get('business_contact_method')
        phone_number = data['graphql']['user'].get('phone_number')
        email = data['graphql']['user'].get('business_email')
        
        print(f"Username: {usrname}")
        print(f"Business Contact Method: {business_contact}")
        print(f"Phone Number: {phone_number if phone_number else 'Not Available'}")
        print(f"Email: {email if email else 'Not Available'}")
    except KeyError:
        print("No business contact information found.")
else:
    print(f"Failed to fetch profile data. Status code: {response.status_code}")


Failed to fetch profile data. Status code: 201


In [2]:
# Import instaloader package and regex
import instaloader
import re

# creating an Instaloader() object
ig = instaloader.Instaloader()

# Taking the Instagram username as input
usrname = "fitwtanishq"

# Fetching the profile details using the Instaloader object
profile = instaloader.Profile.from_username(ig.context, usrname)

# Printing the fetched details and storing the profile picture
print("Username:", profile.username)
print("Number of Posts Uploaded:", profile.mediacount)
print(profile.username + " has " + str(profile.followers) + " followers.")
print(profile.username + " is following " + str(profile.followees) + " people")
print("Bio:", profile.biography)

# Attempting to extract the mobile number from the bio using regex
phone_number = re.findall(r'\+?\d{1,4}[-.\s]??\d{3}[-.\s]??\d{3,4}[-.\s]??\d{3,4}', profile.biography)
if phone_number:
    print("Mobile Number(s):", ', '.join(phone_number))
else:
    print("No mobile number found in the bio.")

# Downloading the profile picture
instaloader.Instaloader().download_profile(usrname, profile_pic_only=True)


Username: fitwtanishq
Number of Posts Uploaded: 15
fitwtanishq has 266 followers.
fitwtanishq is following 207 people
Bio: Hello “YOU”
No mobile number found in the bio.
fitwtanishq\2025-01-16_18-56-30_UTC_profile_pic.jpg already exists


### information extraction

In [7]:
import pandas as pd
import csv
import re
import time
import requests
from bs4 import BeautifulSoup
from instagrapi import Client

# Instagram login credentials
USERNAME = "gigi.chauhaan"
PASSWORD = "Tanseem@2004"  # Replace with actual password

# Initialize the Client
cl = Client()

# Login to Instagram
try:
    cl.login(USERNAME, PASSWORD)
    print("✅ Login successful!")
except Exception as e:
    print(f"❌ Login failed: {e}")
    exit()  # Stop execution if login fails

# Read the input CSV file containing usernames
df = pd.read_csv('users.csv')

# Output CSV file
output_file = 'scraped_instagram_data_contact.csv'

# Check if the file already exists to decide whether to write a header
write_header = not pd.io.common.file_exists(output_file)

with open(output_file, 'a', newline='', encoding='utf-8') as file:
    csv_writer = csv.writer(file)

    # Write header if new file
    if write_header:
        csv_writer.writerow(['Username', 'Followers', 'Bio', 'Website', 'Extracted Email', 'Extracted Phone'])

    # Iterate over usernames
    for index, row in df.iterrows():
        username = row['usernames']  # Ensure your CSV has a 'Username' column

        try:
            # Fetch user details
            user_id = cl.user_id_from_username(username)
            user_info = cl.user_info(user_id)

            followers = user_info.follower_count
            bio = user_info.biography  # Bio text
            website = user_info.external_url  # Website link (if available)

            # Extract email from bio (if available)
            email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
            extracted_email = re.findall(email_pattern, bio)
            email = extracted_email[0] if extracted_email else "Not Found"

            # Extract phone number from bio (if available)
            phone_pattern = r"\+?\d[\d -]{8,14}\d"
            extracted_phone = re.findall(phone_pattern, bio)
            phone_number = extracted_phone[0] if extracted_phone else "Not Found"

            # If the user has a website, try scraping for contact info
            if website:
                try:
                    headers = {"User-Agent": "Mozilla/5.0"}
                    response = requests.get(website, headers=headers, timeout=5)
                    soup = BeautifulSoup(response.text, 'html.parser')

                    # Extract emails from the website
                    website_emails = re.findall(email_pattern, soup.text)
                    if website_emails:
                        email = website_emails[0]

                    # Extract phone numbers from the website
                    website_phones = re.findall(phone_pattern, soup.text)
                    if website_phones:
                        phone_number = website_phones[0]

                except Exception as e:
                    print(f"⚠️ Error scraping website {website}: {e}")

            # Print and save data
            print(f"✅ {username}: {followers} followers | Email: {email} | Phone: {phone_number}")
            csv_writer.writerow([username, followers, bio, website, email, phone_number])

            # Sleep to avoid rate-limiting
            time.sleep(2)

        except Exception as e:
            print(f"❌ Error fetching data for {username}: {e}")


✅ Login successful!


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/surabhi_agrawal_patni/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/surabhi_agrawal_patni/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/surabhi_agrawal_patni/?__a=1&__d=dis) >>> 


✅ surabhi_agrawal_patni: 6696 followers | Email: surabhisagr@gmail.com | Phone: Not Found


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/i_am_piku_09/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/i_am_piku_09/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/i_am_piku_09/?__a=1&__d=dis) >>> 


✅ i_am_piku_09: 978 followers | Email: Not Found | Phone: Not Found


...

✅ eef_finds: 408 followers | Email: Not Found | Phone: Not Found


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/makeup_by_manii/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/makeup_by_manii/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/makeup_by_manii/?__a=1&__d=dis) >>> 


✅ makeup_by_manii: 12779 followers | Email: Not Found | Phone: Not Found


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/layla_tahreem/?__a=1&__d=dis) >>> 


KeyboardInterrupt: 

In [8]:
import pandas as pd
import csv
import re
import time
import requests
from bs4 import BeautifulSoup
from instagrapi import Client

# Instagram login credentials
USERNAME = "gigi.chauhaan"
PASSWORD = "Tanseem@2004"  # Replace with actual password

# Initialize the Client
cl = Client()

# Login to Instagram
try:
    cl.login(USERNAME, PASSWORD)
    print("✅ Login successful!")
except Exception as e:
    print(f"❌ Login failed: {e}")
    exit()  # Stop execution if login fails

# Read the input CSV file containing usernames
df = pd.read_csv('users.csv')

# Output CSV file
output_file = 'scraped_instagram_data.csv'

# Ensure the CSV file has headers if it doesn't exist
try:
    with open(output_file, 'x', newline='', encoding='utf-8') as file:
        csv_writer = csv.writer(file)
        csv_writer.writerow(['Username', 'Followers', 'Bio', 'Website', 'Extracted Email', 'Extracted Phone'])
except FileExistsError:
    pass  # File already exists, no need to create headers

# Open the output file in append mode to save data after each user
with open(output_file, 'a', newline='', encoding='utf-8') as file:
    csv_writer = csv.writer(file)

    # Iterate over usernames
    for index, row in df.iterrows():
        username = row['usernames']  # Ensure your CSV has a 'Username' column

        try:
            # Fetch user details
            user_id = cl.user_id_from_username(username)
            user_info = cl.user_info(user_id)

            followers = user_info.follower_count
            bio = user_info.biography  # Bio text
            website = user_info.external_url  # Website link (if available)

            # Extract email from bio (if available)
            email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
            extracted_email = re.findall(email_pattern, bio)
            email = extracted_email[0] if extracted_email else "Not Found"

            # Extract phone number from bio (if available)
            phone_pattern = r"\+?\d[\d -]{8,14}\d"
            extracted_phone = re.findall(phone_pattern, bio)
            phone_number = extracted_phone[0] if extracted_phone else "Not Found"

            # If the user has a website, try scraping for contact info
            if website:
                try:
                    headers = {"User-Agent": "Mozilla/5.0"}
                    response = requests.get(website, headers=headers, timeout=5)
                    soup = BeautifulSoup(response.text, 'html.parser')

                    # Extract emails from the website
                    website_emails = re.findall(email_pattern, soup.text)
                    if website_emails:
                        email = website_emails[0]

                    # Extract phone numbers from the website
                    website_phones = re.findall(phone_pattern, soup.text)
                    if website_phones:
                        phone_number = website_phones[0]

                except Exception as e:
                    print(f"⚠️ Error scraping website {website}: {e}")

            # Print and save data immediately
            print(f"✅ {username}: {followers} followers | Email: {email} | Phone: {phone_number}")

            # Write to CSV after each user to prevent data loss
            csv_writer.writerow([username, followers, bio, website, email, phone_number])
            file.flush()  # Ensure data is written immediately

            # Sleep to avoid rate-limiting
            time.sleep(2)

        except Exception as e:
            print(f"❌ Error fetching data for {username}: {e}")


✅ Login successful!
✅ surabhi_agrawal_patni: 6696 followers | Email: surabhisagr@gmail.com | Phone: Not Found
✅ i_am_piku_09: 978 followers | Email: Not Found | Phone: Not Found
✅ trisshhhaa_: 1049 followers | Email: trishakapoor605@gmail.com | Phone: Not Found
✅ soumyatyagii: 33 followers | Email: tyagi.somya12@gmail.com | Phone: Not Found
✅ i.aashusinghh: 156 followers | Email: aashubiz2003@gmail.com | Phone: Not Found
✅ sweet_whiskey__: 27218 followers | Email: Not Found | Phone: Not Found
✅ eef_finds: 408 followers | Email: Not Found | Phone: Not Found
✅ makeup_by_manii: 12779 followers | Email: Not Found | Phone: Not Found


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/layla_tahreem/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/layla_tahreem/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/layla_tahreem/?__a=1&__d=dis) >>> 


✅ layla_tahreem: 919 followers | Email: lovelaylaalfiya@gmail.com | Phone: Not Found


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/glamchic_vibes/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/glamchic_vibes/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/glamchic_vibes/?__a=1&__d=dis) >>> 


✅ glamchic_vibes: 72 followers | Email: Not Found | Phone: Not Found


...

❌ Error fetching data for happy_boy_adil6007: login_required


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/__finds4u/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/__finds4u/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/__finds4u/?__a=1&__d=dis) >>> 


❌ Error fetching data for __finds4u: login_required


Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/jollythoudam/?__a=1&__d=dis) >>> 
Status 201: JSONDecodeError in public_request (url=https://www.instagram.com/jollythoudam/?__a=1&__d=dis) >>> 


KeyboardInterrupt: 

In [1]:
import pandas as pd
data=pd.read_csv("20feb.csv")
print(len(data))

52499


In [3]:
import pandas as pd
import re
import os

# Input and output file paths
input_csv = "20feb.csv"
output_csv = "users.csv"

# Check if the input file exists
if not os.path.exists(input_csv):
    print(f"❌ Error: The input file '{input_csv}' does not exist. Please ensure the file is in the correct directory.")
else:
    try:
        # Load the CSV file
        df = pd.read_csv(input_csv)

        # Ensure 'tagged_links' column exists
        if "tagged_links" not in df.columns:
            raise ValueError("The CSV file does not contain the 'tagged_links' column.")

        # Regular expression to extract usernames
        pattern = r"https://www\.instagram\.com/([^/]+)/"

        # Optional: Process a subset of rows (e.g., from row 964 onward)
        data_r = df["tagged_links"]

        # Extract usernames and create a new DataFrame with only usernames
        usernames = data_r.apply(
            lambda link: re.search(pattern, link).group(1) if pd.notna(link) and re.search(pattern, link) else None
        )
        usernames_df = pd.DataFrame({"usernames": usernames})

        # Save the usernames DataFrame to a new CSV file
        usernames_df.to_csv(output_csv, index=False)
        print(f"✅ Usernames extracted and saved to {output_csv}")

        # Load and display the total number of usernames
        if os.path.exists(output_csv):
            data = pd.read_csv(output_csv)
            print(f"📄 Total unique usernames saved: {len(data)}")
        else:
            print("❌ Error: The output file was not created successfully.")

    except Exception as e:
        print(f"❌ Error: {e}")


✅ Usernames extracted and saved to users.csv
📄 Total unique usernames saved: 52500


In [4]:
import pandas as pd

# File path for the CSV file
input_csv = "users.csv"

try:
    # Load the CSV file
    df = pd.read_csv(input_csv)

    # Ensure 'usernames' column exists
    if "usernames" not in df.columns:
        raise ValueError("The CSV file does not contain the 'usernames' column.")

    # Total usernames before removing duplicates
    total_before = len(df)
    print(f"📄 Total usernames before removing duplicates: {total_before}")

    # Remove duplicate usernames
    df_unique = df.drop_duplicates(subset="usernames")

    # Total usernames after removing duplicates
    total_after = len(df_unique)
    print(f"✅ Total unique usernames: {total_after}")

    # Save the updated DataFrame back to the same file
    df_unique.to_csv(input_csv, index=False)
    print(f"✅ Unique usernames saved back to {input_csv}")

except Exception as e:
    print(f"❌ Error: {e}")


📄 Total usernames before removing duplicates: 52500
✅ Total unique usernames: 24712
✅ Unique usernames saved back to users.csv


In [ ]:
import csv
import instaloader

# Create an Instaloader object
ig = instaloader.Instaloader()

# Function to fetch Instagram details for a given username
def fetch_instagram_details(username):
    try:
        # Fetch profile details using instaloader
        profile = instaloader.Profile.from_username(ig.context, username)

        # Collecting post URLs
        post_links = [post.url for post in profile.get_posts()]

        # Storing the fetched details
        user_data = {
            'Username': profile.username,
            'Number of Posts': profile.mediacount,
            'Followers': profile.followers,
            'Following': profile.followees,
            'Bio': profile.biography,
            'Post Links': ", ".join(post_links)  # Joining post URLs in a single string
        }

        # Return the user data
        return user_data

    except Exception as e:
        print(f"❌ Error fetching details for {username}: {e}")
        return None

# Read usernames from CSV file and fetch details
def fetch_from_csv(csv_file, output_file):
    with open(csv_file, mode='r', encoding='utf-8') as infile:
        csv_reader = csv.reader(infile)

        # Skip the header (if any)
        next(csv_reader)

        # Open the output CSV file to write the details
        with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
            fieldnames = ['Username', 'Number of Posts', 'Followers', 'Following', 'Bio', 'Post Links']
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()

            # Loop through each row in the CSV file
            for row in csv_reader:
                if row:  # Check if the row is not empty
                    username = row[0]  # Assuming the username is in the first column
                    user_data = fetch_instagram_details(username)
                    if user_data:
                        writer.writerow(user_data)
                        print(f"Details for {username} saved successfully.\n")

# Main execution
csv_file = 'only_usernames.csv'  # Replace with your actual CSV file path
output_file = 'instagram_details.csv'  # File to save the fetched details
fetch_from_csv(csv_file, output_file)


Details for fonic_id saved successfully.

Details for dede.dede_14 saved successfully.

Details for trapphhousee saved successfully.

Details for que_fonic saved successfully.

Details for maryam_mohajervapesaran saved successfully.


Too many queries in the last time. Need to wait 166 seconds, until 13:43.
Details for mmint_titirat saved successfully.

Details for alhanb.music saved successfully.

Details for fonicoficial saved successfully.

Details for ____.narcisa.____ saved successfully.

Details for fidullmusic saved successfully.


Too many queries in the last time. Need to wait 109 seconds, until 13:54.
Details for aidangillen.russia saved successfully.

Details for e_lhombre_fuerte saved successfully.



In [12]:
d3=pd.read_csv("instagram_details.csv")
d3

,Username,Number of Posts,Followers,Following,Bio
0,fonic_id,30,22,19,Electrical Accesories with SNI approval || fon...
1,dede.dede_14,76,105,182,Oradea.\nBox.\n14 ani.\n❤😍
2,trapphhousee,967,310,39,Content despre artiști\n 🎶❤️\n@1._rms\n@raul0s
3,que_fonic,621,1467,1577,All about the CRYSTAL LIFE 💎\nDownload #LowKey...
4,maryam_mohajervapesaran,461,4582,2053,@BAFTA Winner best British short animation. \n...
5,mmint_titirat,855,3663,1407,🐝 𝙼 𝙸 𝙽 𝚃\n💊 𝙿 𝚑 𝚊 𝚛 𝚖 𝚊 𝚌 𝚒 𝚜 𝚝\n💜 𝙲 𝙼 𝚄 ’𝟻 𝟸...
6,alhanb.music,53,1107,697,Music Student 🦋
7,fonicoficial,21,395,3,NaN
8,____.narcisa.____,15,93,289,♡13\n♡Iulie\n♡Satra B.E.N.Z\n♡OSOD II\n♡#KILLA...
9,fidullmusic,609,10168,4785,"Underground house music magazine, record label..."


In [1]:
import requests
import pandas as pd

# Define API endpoint and parameters
ACCESS_TOKEN = "EAAdi24rFvZCYBO3xtcw4zjZBknk8r3D4WolcdoKMgNDuOsQo2hQAul55xxZC3ZCRZAH8TL45ClShsHnswAOCAygZBD2XorEuR4S78lNWYZBYFa3qY4ZATTul48bNMov0qWZCvbqn3fZBZC0jHZAEJyh8cZAOon2tvpqbpMvaMV7xZAxtY92UWwrzpTdjTjENxZBR2XZBm644jMdG93mhwIaftvfomUcUHGd9GQZDZD"  # Replace with your valid access token
MEDIA_ID = "17841563551082812"
USER_ID = "17841468142856047"
FIELDS = "id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id"
LIMIT = 500
API_URL = f"https://graph.facebook.com/v22.0/{MEDIA_ID}/recent_media?fields={FIELDS}&limit={LIMIT}&user_id={USER_ID}&access_token={ACCESS_TOKEN}"

# Make the request to the Instagram Graph API
response = requests.get(API_URL)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Extract relevant data
    media_list = []
    for media in data.get("data", []):
        media_list.append({
            "id": media.get("id"),
            "caption": media.get("caption"),
            "media_type": media.get("media_type"),
            "media_url": media.get("media_url"),
            "permalink": media.get("permalink"),
            "timestamp": media.get("timestamp"),
            "comments_count": media.get("comments_count"),
            "like_count": media.get("like_count"),
            "legacy_instagram_media_id": media.get("legacy_instagram_media_id")
        })

    # Convert data to a DataFrame and save it to CSV
    df = pd.DataFrame(media_list)
    df.to_csv("fetchd_instapi.csv", index=False)

    print("Data successfully fetched and saved to fetchd_instapi.csv.")

else:
    print(f"Error fetching data: {response.json()}")


Data successfully fetched and saved to fetchd_instapi.csv.


In [2]:
d=pd.read_csv("fetchd_instapi.csv")
d

,id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id
0,18053612332893807,Continue reading👇🏽I PROMISE this caption is wo...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfLBXqzHrZ/,2025-01-31T10:44:51+0000,2,25.0,9.528994e+15
1,18031420610262498,"When you need that ‘just-had-a-facial glow’, r...",VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfJ2KKJlXt/,2025-01-31T10:35:04+0000,2,NaN,8.762041e+15
2,18160731394331742,Glow starts with the right ingredients! ✨ Targ...,CAROUSEL_ALBUM,https://scontent.cdninstagram.com/v/t51.75761-...,https://www.instagram.com/p/DFfI355SkLT/,2025-01-31T10:25:40+0000,0,NaN,9.360669e+15
3,18096277453511885,Skincare Tool!!\n- it has 5 different attachme...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfIbOXIZBT/,2025-01-31T10:22:06+0000,0,2.0,9.009855e+15
4,18048852044329185,TRETIHEAL 0.05% AVAILABLE . \nDM for Order . \...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfBYDwOhwR/,2025-01-31T09:22:40+0000,0,1.0,9.054128e+15
5,18031484600254514,Wake up in the glowing hour by letting the Pon...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFe44yZTX9F/,2025-01-31T08:08:36+0000,1,27.0,9.029003e+15
6,18153301810351729,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,NaN,https://www.instagram.com/reel/DFe0eNGx7RY/,2025-01-31T07:31:12+0000,5,61.0,9.445434e+15
7,18068896762837788,"I’m a sucker for lip balms, and so I finally d...",VIDEO,NaN,https://www.instagram.com/reel/DFezHLWyJys/,2025-01-31T07:21:09+0000,0,1.0,8.903277e+15
8,17945968679943484,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrgntzAf/,2025-01-31T06:47:41+0000,9,265.0,2.813987e+16
9,18121955293418975,Looking for glowing skin without or with makeu...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrekPyxr/,2025-01-31T06:46:15+0000,64,NaN,9.591496e+15


In [9]:
import requests
import pandas as pd

# Define API endpoint and parameters
ACCESS_TOKEN = "EAAdi24rFvZCYBO3xtcw4zjZBknk8r3D4WolcdoKMgNDuOsQo2hQAul55xxZC3ZCRZAH8TL45ClShsHnswAOCAygZBD2XorEuR4S78lNWYZBYFa3qY4ZATTul48bNMov0qWZCvbqn3fZBZC0jHZAEJyh8cZAOon2tvpqbpMvaMV7xZAxtY92UWwrzpTdjTjENxZBR2XZBm644jMdG93mhwIaftvfomUcUHGd9GQZDZD"  # Replace with your valid access token
MEDIA_ID = "17841563551082812"
USER_ID = "17841468142856047"
FIELDS = "id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id"
LIMIT = 500
API_URL = f"https://graph.facebook.com/v20.0/{MEDIA_ID}/recent_media?fields={FIELDS}&limit={LIMIT}&user_id={USER_ID}&access_token={ACCESS_TOKEN}"

# Make the request to the Instagram Graph API
response = requests.get(API_URL)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Extract relevant data
    media_list = []
    for media in data.get("data", []):
        media_list.append({
            "id": media.get("id"),
            "caption": media.get("caption"),
            "media_type": media.get("media_type"),
            "media_url": media.get("media_url"),
            "permalink": media.get("permalink"),
            "timestamp": media.get("timestamp"),
            "comments_count": media.get("comments_count"),
            "like_count": media.get("like_count"),
            "legacy_instagram_media_id": media.get("legacy_instagram_media_id"),
            "ig_id": media.get("ig_id"),
            "iowner": media.get("owner"),
            "user_name": media.get("user_name")
        })

    # Convert data to a DataFrame and save it to CSV
    df = pd.DataFrame(media_list)
    df.to_csv("fetchd_instapi.csv", index=False)

    print("Data successfully fetched and saved to fetchd_instapi.csv.")

else:
    print(f"Error fetching data: {response.json()}")
d=pd.read_csv("fetchd_instapi.csv")
d

Data successfully fetched and saved to fetchd_instapi.csv.


,id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id,ig_id,iowner,user_name
0,18053612332893807,Continue reading👇🏽I PROMISE this caption is wo...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfLBXqzHrZ/,2025-01-31T10:44:51+0000,2,25.0,9.528994e+15,NaN,NaN,NaN
1,18031420610262498,"When you need that ‘just-had-a-facial glow’, r...",VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfJ2KKJlXt/,2025-01-31T10:35:04+0000,2,NaN,8.762041e+15,NaN,NaN,NaN
2,18160731394331742,Glow starts with the right ingredients! ✨ Targ...,CAROUSEL_ALBUM,https://scontent.cdninstagram.com/v/t51.75761-...,https://www.instagram.com/p/DFfI355SkLT/,2025-01-31T10:25:40+0000,0,NaN,9.360669e+15,NaN,NaN,NaN
3,18096277453511885,Skincare Tool!!\n- it has 5 different attachme...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfIbOXIZBT/,2025-01-31T10:22:06+0000,0,2.0,9.009855e+15,NaN,NaN,NaN
4,18048852044329185,TRETIHEAL 0.05% AVAILABLE . \nDM for Order . \...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfBYDwOhwR/,2025-01-31T09:22:40+0000,0,1.0,9.054128e+15,NaN,NaN,NaN
5,18031484600254514,Wake up in the glowing hour by letting the Pon...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFe44yZTX9F/,2025-01-31T08:08:36+0000,1,27.0,9.029003e+15,NaN,NaN,NaN
6,18153301810351729,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,NaN,https://www.instagram.com/reel/DFe0eNGx7RY/,2025-01-31T07:31:12+0000,5,61.0,9.445434e+15,NaN,NaN,NaN
7,18068896762837788,"I’m a sucker for lip balms, and so I finally d...",VIDEO,NaN,https://www.instagram.com/reel/DFezHLWyJys/,2025-01-31T07:21:09+0000,0,1.0,8.903277e+15,NaN,NaN,NaN
8,17945968679943484,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrgntzAf/,2025-01-31T06:47:41+0000,9,269.0,2.813987e+16,NaN,NaN,NaN
9,18121955293418975,Looking for glowing skin without or with makeu...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrekPyxr/,2025-01-31T06:46:15+0000,65,NaN,9.591496e+15,NaN,NaN,NaN


In [10]:
import requests
import pandas as pd

# Define API endpoint and parameters
ACCESS_TOKEN = "EAAdi24rFvZCYBO3xtcw4zjZBknk8r3D4WolcdoKMgNDuOsQo2hQAul55xxZC3ZCRZAH8TL45ClShsHnswAOCAygZBD2XorEuR4S78lNWYZBYFa3qY4ZATTul48bNMov0qWZCvbqn3fZBZC0jHZAEJyh8cZAOon2tvpqbpMvaMV7xZAxtY92UWwrzpTdjTjENxZBR2XZBm644jMdG93mhwIaftvfomUcUHGd9GQZDZD"  # Replace with your valid access token
MEDIA_ID = "17841563551082812"
USER_ID = "17841468142856047"
FIELDS = "id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id"
LIMIT = 500
API_URL = f"https://graph.facebook.com/v16.0/17841563551082812?fields=id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,owner{username}&access_token=YOUR_ACCESS_TOKEN"

# Make the request to the Instagram Graph API
response = requests.get(API_URL)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Extract relevant data
    media_list = []
    for media in data.get("data", []):
        media_list.append({
    "id": media.get("id"),
    "caption": media.get("caption"),
    "media_type": media.get("media_type"),
    "media_url": media.get("media_url"),
    "permalink": media.get("permalink"),
    "timestamp": media.get("timestamp"),
    "comments_count": media.get("comments_count"),
    "like_count": media.get("like_count"),
    "user_name": media.get("owner", {}).get("username")  # Ensures safe access to username
       })

    # Convert data to a DataFrame and save it to CSV
    df = pd.DataFrame(media_list)
    df.to_csv("fetchd_instapi.csv", index=False)

    print("Data successfully fetched and saved to fetchd_instapi.csv.")

else:
    print(f"Error fetching data: {response.json()}")
d=pd.read_csv("fetchd_instapi.csv")
d

NameError: name 'username' is not defined

In [12]:
import requests
import pandas as pd

# Define API endpoint and parameters
ACCESS_TOKEN = "EAAdi24rFvZCYBO3xtcw4zjZBknk8r3D4WolcdoKMgNDuOsQo2hQAul55xxZC3ZCRZAH8TL45ClShsHnswAOCAygZBD2XorEuR4S78lNWYZBYFa3qY4ZATTul48bNMov0qWZCvbqn3fZBZC0jHZAEJyh8cZAOon2tvpqbpMvaMV7xZAxtY92UWwrzpTdjTjENxZBR2XZBm644jMdG93mhwIaftvfomUcUHGd9GQZDZD"  # Replace with your valid access token  # Replace with your valid access token
MEDIA_ID = "17841563551082812"

# Corrected API URL (fetching only owner.username)
API_URL = f"https://graph.facebook.com/v16.0/{MEDIA_ID}?fields=owner{{username}}&access_token={ACCESS_TOKEN}"

# Make the request to the Instagram Graph API
response = requests.get(API_URL)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Extract the owner username
    owner_username = data.get("owner", {}).get("username")

    # Save data to a CSV file
    df = pd.DataFrame([{"media_id": MEDIA_ID, "owner_username": owner_username}])
    df.to_csv("fetchd_instapi.csv", index=False)

    print("Owner username successfully fetched and saved to fetchd_instapi.csv.")

else:
    print(f"Error fetching data: {response.json()}")

# Load and display the saved data
d = pd.read_csv("fetchd_instapi.csv")
d


Error fetching data: {'error': {'message': '(#100) Tried accessing nonexisting field (owner) on node type (ShadowIGHashtag)', 'type': 'OAuthException', 'code': 100, 'fbtrace_id': 'ABHgoSvTOuKc6VQtDqADIdY'}}


,id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id,ig_id,iowner,user_name
0,18053612332893807,Continue reading👇🏽I PROMISE this caption is wo...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfLBXqzHrZ/,2025-01-31T10:44:51+0000,2,25.0,9.528994e+15,NaN,NaN,NaN
1,18031420610262498,"When you need that ‘just-had-a-facial glow’, r...",VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfJ2KKJlXt/,2025-01-31T10:35:04+0000,2,NaN,8.762041e+15,NaN,NaN,NaN
2,18160731394331742,Glow starts with the right ingredients! ✨ Targ...,CAROUSEL_ALBUM,https://scontent.cdninstagram.com/v/t51.75761-...,https://www.instagram.com/p/DFfI355SkLT/,2025-01-31T10:25:40+0000,0,NaN,9.360669e+15,NaN,NaN,NaN
3,18096277453511885,Skincare Tool!!\n- it has 5 different attachme...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfIbOXIZBT/,2025-01-31T10:22:06+0000,0,2.0,9.009855e+15,NaN,NaN,NaN
4,18048852044329185,TRETIHEAL 0.05% AVAILABLE . \nDM for Order . \...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfBYDwOhwR/,2025-01-31T09:22:40+0000,0,1.0,9.054128e+15,NaN,NaN,NaN
5,18031484600254514,Wake up in the glowing hour by letting the Pon...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFe44yZTX9F/,2025-01-31T08:08:36+0000,1,27.0,9.029003e+15,NaN,NaN,NaN
6,18153301810351729,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,NaN,https://www.instagram.com/reel/DFe0eNGx7RY/,2025-01-31T07:31:12+0000,5,61.0,9.445434e+15,NaN,NaN,NaN
7,18068896762837788,"I’m a sucker for lip balms, and so I finally d...",VIDEO,NaN,https://www.instagram.com/reel/DFezHLWyJys/,2025-01-31T07:21:09+0000,0,1.0,8.903277e+15,NaN,NaN,NaN
8,17945968679943484,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrgntzAf/,2025-01-31T06:47:41+0000,9,269.0,2.813987e+16,NaN,NaN,NaN
9,18121955293418975,Looking for glowing skin without or with makeu...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrekPyxr/,2025-01-31T06:46:15+0000,65,NaN,9.591496e+15,NaN,NaN,NaN


In [19]:
import requests
import pandas as pd

# Define API endpoint and parameters
MEDIA_ID = "17841563551082812"
FIELDS = "id,owner,timestamp"
API_URL = f"https://graph.facebook.com/v16.0/{MEDIA_ID}?fields={FIELDS}&access_token={ACCESS_TOKEN}"

# Make the request to the Instagram Graph API
response = requests.get(API_URL)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Extract relevant data
    media_info = {
        "id": data.get("id"),
        # "media_type": data.get("media_type"),
        # "media_url": data.get("media_url"),
        "timestamp": data.get("timestamp"),
        "owner": data.get("owner", {}).get("username")  # Ensure safe access to the owner's username
    }

    # Save data to CSV
    df = pd.DataFrame([media_info])
    df.to_csv("media_info.csv", index=False)

    print("Media data successfully fetched and saved to media_info.csv.")
else:
    print(f"Error fetching data: {response.json()}")

# Display the fetched data
# d = pd.read_csv("media_info.csv")
# print(d)


Error fetching data: {'error': {'message': '(#100) Tried accessing nonexisting field (owner) on node type (ShadowIGHashtag)', 'type': 'OAuthException', 'code': 100, 'fbtrace_id': 'A8IujwLs4a45r4w3rnG8xUl'}}


In [20]:
import requests
import pandas as pd

# Define API endpoint and parameters
ACCESS_TOKEN = "EAAdi24rFvZCYBO3xtcw4zjZBknk8r3D4WolcdoKMgNDuOsQo2hQAul55xxZC3ZCRZAH8TL45ClShsHnswAOCAygZBD2XorEuR4S78lNWYZBYFa3qY4ZATTul48bNMov0qWZCvbqn3fZBZC0jHZAEJyh8cZAOon2tvpqbpMvaMV7xZAxtY92UWwrzpTdjTjENxZBR2XZBm644jMdG93mhwIaftvfomUcUHGd9GQZDZD"  # Replace with your valid access token
MEDIA_ID = "17841563551082812"
USER_ID = "17841468142856047"
FIELDS = "id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id"
LIMIT = 500

# Construct API URL
API_URL = f"https://graph.facebook.com/v22.0/{MEDIA_ID}/recent_media?fields={FIELDS}&limit={LIMIT}&user_id={USER_ID}&access_token={ACCESS_TOKEN}"

# Make the request to the Instagram Graph API
response = requests.get(API_URL)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Extract relevant media data
    media_list = []
    for media in data.get("data", []):
        media_list.append({
            "id": media.get("id"),
            "caption": media.get("caption"),
            "media_type": media.get("media_type"),
            "media_url": media.get("media_url"),
            "permalink": media.get("permalink"),
            "timestamp": media.get("timestamp"),
            "comments_count": media.get("comments_count"),
            "like_count": media.get("like_count"),
            "legacy_instagram_media_id": media.get("legacy_instagram_media_id")
        })

    # Convert the data into a DataFrame and save it to a CSV file
    df = pd.DataFrame(media_list)
    df.to_csv("fetchd_instapi.csv", index=False)

    print("Data successfully fetched and saved to fetchd_instapi.csv.")

else:
    print(f"Error fetching data: {response.json()}")


Data successfully fetched and saved to fetchd_instapi.csv.


In [21]:
df=pd.read_csv("fetchd_instapi.csv")
df

,id,caption,media_type,media_url,permalink,timestamp,comments_count,like_count,legacy_instagram_media_id
0,18053612332893807,Continue reading👇🏽I PROMISE this caption is wo...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfLBXqzHrZ/,2025-01-31T10:44:51+0000,3,28.0,9.528994e+15
1,18031420610262498,"When you need that ‘just-had-a-facial glow’, r...",VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfJ2KKJlXt/,2025-01-31T10:35:04+0000,4,NaN,8.762041e+15
2,18160731394331742,Glow starts with the right ingredients! ✨ Targ...,CAROUSEL_ALBUM,https://scontent.cdninstagram.com/v/t51.75761-...,https://www.instagram.com/p/DFfI355SkLT/,2025-01-31T10:25:40+0000,0,NaN,9.360669e+15
3,18096277453511885,Skincare Tool!!\n- it has 5 different attachme...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfIbOXIZBT/,2025-01-31T10:22:06+0000,0,2.0,9.009855e+15
4,18048852044329185,TRETIHEAL 0.05% AVAILABLE . \nDM for Order . \...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFfBYDwOhwR/,2025-01-31T09:22:40+0000,0,1.0,9.054128e+15
5,18031484600254514,Wake up in the glowing hour by letting the Pon...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFe44yZTX9F/,2025-01-31T08:08:36+0000,1,27.0,9.029003e+15
6,18153301810351729,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,NaN,https://www.instagram.com/reel/DFe0eNGx7RY/,2025-01-31T07:31:12+0000,5,63.0,9.445434e+15
7,18068896762837788,"I’m a sucker for lip balms, and so I finally d...",VIDEO,NaN,https://www.instagram.com/reel/DFezHLWyJys/,2025-01-31T07:21:09+0000,0,1.0,8.903277e+15
8,17945968679943484,My skin’s best-kept secret! 😉 Not gatekeeping ...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrgntzAf/,2025-01-31T06:47:41+0000,9,277.0,2.813987e+16
9,18121955293418975,Looking for glowing skin without or with makeu...,VIDEO,https://instagram.fdel52-1.fna.fbcdn.net/o1/v/...,https://www.instagram.com/reel/DFevrekPyxr/,2025-01-31T06:46:15+0000,67,NaN,9.591496e+15


In [34]:
import requests

# Replace with your access token
# ACCESS_TOKEN = 'your-access-token'

# List of media IDs from the result
media_ids = [18053612332893807]

def get_owner_details(media_id):
    # Get the owner ID for a media post
    url = f'https://graph.facebook.com/v22.0/{media_id}?fields=media_url&access_token=EAAdi24rFvZCYBO2J1Kb55JrkDBFQryDzf9kR2xiti1ynQjYKf2zEAhXavcDSWyvAM3BJK9IuY7nVajulXtkOkj08h2pdvkq4fGvFrcZCVUl5buDZCwiAiIG9VkTqDpyHbIsLIYiPFXMVUP27OP2DkQ8WMVOCGvepePwUN9SFaJVASuwPx6Iezy15Tk1aHr4jZCHjBNwHmRry5IZCnGFmgxmiozAZDZD'
    response = requests.get(url)
    data2 = response.json()
    
    # # Fetch the owner's details using their user ID
    # if 'owner' in data:
    #     owner_id = data['owner']['id']
    #     owner_url = f'https://graph.facebook.com/v16.0/{owner_id}?fields=id,username,profile_picture_url&access_token={ACCESS_TOKEN}'
    #     owner_response = requests.get(owner_url)
    #     owner_data = owner_response.json()
        
    #     return owner_data
    # else:
    #     return None
    return data2

# Fetch owner details for each media ID
# owner_details = [get_owner_details(media_id) for media_id in media_ids]
# print(owner_details)
data2=get_owner_details(18053612332893807)
data2

{'error': {'message': "Unsupported get request. Object with ID '18053612332893807' does not exist, cannot be loaded due to missing permissions, or does not support this operation. Please read the Graph API documentation at https://developers.facebook.com/docs/graph-api",
  'type': 'GraphMethodException',
  'code': 100,
  'error_subcode': 33,
  'fbtrace_id': 'AVz8RPy1hcS4SH4j3O0ZJpL'}}